# 20_01_テンソルを作る_計算する_形を変える_NumPy

コード：`ch20/blocks/20_01_テンソルを作る_計算する_形を変える_NumPy.ipynb`

本文のコードをコードセルで直接実行します。

上からセルを実行してください。ローカルではコード一式と同じPython環境をJupyterのカーネルに選びます。Colabでコード一式が必要な場合は、準備セルで配布ZIPをアップロードします。実行確認の範囲は `VALIDATION_COLAB_20261110.md` を参照してください。


In [ ]:
import subprocess
import sys

# 既に導入済みなら不要な更新をせず、満たしていない依存だけを導入する。
subprocess.run([sys.executable, "-m", "pip", "install", *['numpy>=1.26,<3', 'torch>=2.6,<3', 'torchvision>=0.21,<1', 'mcp==2.3.0', 'PyYAML==6.0.3', 'opentelemetry-sdk==1.45.2', 'opentelemetry-exporter-otlp-proto-http==1.45.2']], check=True)


In [ ]:
# 付録 A-1：テンソルを作る・形を見る・計算する・NumPy と行き来する
import numpy as np
import torch

x = torch.tensor([[1.0, 2.0, 3.0], [4.0, 5.0, 6.0]])     # Python のリストから
print("形", tuple(x.shape), " 型", x.dtype, " 置き場所", x.device)
z = torch.zeros(2, 3); r = torch.rand(2, 3, generator=torch.Generator().manual_seed(0))
print("zeros", z.tolist(), " rand の 1 行目", [round(v, 3) for v in r[0].tolist()])

# 計算：要素ごとの演算・行列積・ブロードキャスト
print("要素ごとの積", (x * x).tolist())
print("行列積 x @ xᵀ", (x @ x.T).tolist())
print("各行から行の平均を引く", (x - x.mean(dim=1, keepdim=True)).tolist())   # (2,3) − (2,1) → (2,3)

# 形を変える：view は同じメモリを別の形で見る。reshape は必要ならコピーする
v = x.view(3, 2); v[0, 0] = 100.0
print("view を書き換えると元も変わる", x[0, 0].item())
t = x.T                                                    # 転置は連続していない
print("転置は連続か", t.is_contiguous(), "→ view できるか", end=" ")
try:
    t.view(6); print("できる")
except RuntimeError:
    print("できない（reshape か contiguous() を使う）", t.reshape(6).tolist())

# NumPy との行き来：CPU 上では同じメモリを共有する
a = np.ones(3); b = torch.from_numpy(a); b += 1
print("torch 側で足すと NumPy 側も", a.tolist())

# 置き場所を選ぶ：GPU（CUDA）か Apple の GPU（MPS）があれば使い、無ければ CPU
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
print("使える置き場所", device, " 移すと", x.to(device).device)
